# IndexCalc — Phase 6b/6c 테스트

Connection, CovariantDeriv, 자동 전개 (∇ → ∂ + Γ).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from indexcalc import (
    IndexSpace, Tensor, IndexRegistry, parse, to_latex,
    Connection, LeviCivitaConnection,
    CovariantDeriv, covariant, expand_covariant,
    partial,
)
from IPython.display import display, Math

In [2]:
spacetime = IndexSpace("spacetime", dim=4, indices="μνλρσ", metric="g")
lorentz   = IndexSpace("lorentz",   dim=4, indices="abcde", metric="η")

reg = IndexRegistry()
reg.register(spacetime)
reg.register(lorentz)

g     = Tensor("g", [spacetime.lower("μ"), spacetime.lower("ν")])
g_inv = Tensor("g", [spacetime.upper("μ"), spacetime.upper("ν")])

christoffel = LeviCivitaConnection(g, g_inv, spacetime)
spin_conn = Connection("ω", lorentz, deriv_space=spacetime)

## 1. Christoffel symbol 정의

In [3]:
# Γ^μ_νλ = ½ g^{μρ}(∂_ν g_{ρλ} + ∂_λ g_{ρν} - ∂_ρ g_{νλ})
defn = christoffel.definition()
display(Math(r"\Gamma^{\mu}{}_{\nu\lambda} = " + to_latex(defn)))

<IPython.core.display.Math object>

## 2. ∇_μ V^ν (contravariant vector)

In [4]:
V = Tensor("V", [spacetime.upper("ν")])
mu = spacetime.lower("μ")

nabla_V = covariant(V, mu, christoffel)
expanded = expand_covariant(nabla_V)

display(Math(to_latex(nabla_V) + r" \;=\; " + to_latex(expanded)))

<IPython.core.display.Math object>

## 3. ∇_μ V_ν (covariant vector)

In [5]:
V_lower = Tensor("V", [spacetime.lower("ν")])

nabla_Vl = covariant(V_lower, mu, christoffel)
expanded_l = expand_covariant(nabla_Vl)

display(Math(to_latex(nabla_Vl) + r" \;=\; " + to_latex(expanded_l)))

<IPython.core.display.Math object>

## 4. ∇_μ T^ν_λ  (mixed tensor)

In [6]:
T = Tensor("T", [spacetime.upper("ν"), spacetime.lower("λ")])

nabla_T = covariant(T, mu, christoffel)
expanded_T = expand_covariant(nabla_T)

display(Math(to_latex(nabla_T) + r" \;=\; " + to_latex(expanded_T)))

<IPython.core.display.Math object>

## 5. ∇_μ g_νλ (metric compatibility)

In [7]:
g_nl = Tensor("g", [spacetime.lower("ν"), spacetime.lower("λ")])

nabla_g = covariant(g_nl, mu, christoffel)
expanded_g = expand_covariant(nabla_g)

display(Math(to_latex(nabla_g) + r" \;=\; " + to_latex(expanded_g)))
print("↑ Levi-Civita connection이면 이 값은 0")

<IPython.core.display.Math object>

↑ Levi-Civita connection이면 이 값은 0


## 6. 다중 connection: ∇_μ T^{ν a}

spacetime index ν → Christoffel Γ, lorentz index a → spin connection ω

In [8]:
T_mixed = Tensor("T", [spacetime.upper("ν"), lorentz.upper("a")])

connections = {
    spacetime.name: christoffel,
    lorentz.name: spin_conn,
}

nabla_Tm = covariant(T_mixed, mu, connections)
expanded_Tm = expand_covariant(nabla_Tm)

display(Math(to_latex(nabla_Tm) + r" \;=\; " + to_latex(expanded_Tm)))

<IPython.core.display.Math object>

## 7. Spacetime connection만 (lorentz 무시)

In [9]:
nabla_Tm2 = covariant(T_mixed, mu, christoffel)
expanded_Tm2 = expand_covariant(nabla_Tm2)

display(Math(to_latex(nabla_Tm2) + r" \;=\; " + to_latex(expanded_Tm2)))
print("↑ lorentz index a에는 connection 항 없음")

<IPython.core.display.Math object>

↑ lorentz index a에는 connection 항 없음


## 8. Connection 텐서 자체의 LaTeX

In [10]:
# Christoffel symbol
gamma = christoffel.make_tensor("μ", "ν", "λ")
display(gamma)

# Spin connection
omega = spin_conn.make_tensor("a", "μ", "b")
display(omega)

Γ^μ_ν_λ

ω^a_μ_b